# Step 4: WASDE Alignment

This notebook aligns WASDE fundamental data with daily futures data and creates "surprise" features.

**Key concepts:**
- WASDE values are known only on release dates
- Markets react to changes vs expectations (surprises)
- Surprises are standardized (z-scores) for comparability

**Rules:**
- Still NO ML features
- Focus on event alignment and surprise calculation


In [1]:
import pandas as pd
import numpy as np

BASE_PATH = "/Users/aryansinha/Desktop/WASDA"


## STEP 4.2 — Load WASDE Release Calendar

This file is the clock for everything - it tells us when new info enters the market.


In [2]:
# Load WASDE release dates
wasde_dates = pd.read_csv(
    f"{BASE_PATH}/raw/fundamentals/wasde/WASDERELEASEDATES - Sheet1.csv",
    parse_dates=["date"]
)

wasde_dates = wasde_dates.sort_values("date").reset_index(drop=True)

print("WASDE Release Calendar loaded:")
print(f"Total release dates: {len(wasde_dates)}")
print(f"Date range: {wasde_dates['date'].min()} to {wasde_dates['date'].max()}")
print("\nFirst 10 release dates:")
print(wasde_dates.head(10))
print("\nLast 10 release dates:")
print(wasde_dates.tail(10))


WASDE Release Calendar loaded:
Total release dates: 143
Date range: 2015-01-12 00:00:00 to 2026-12-10 00:00:00

First 10 release dates:
        date  is_wasde
0 2015-01-12         1
1 2015-02-10         1
2 2015-03-10         1
3 2015-04-09         1
4 2015-05-12         1
5 2015-06-10         1
6 2015-07-10         1
7 2015-08-12         1
8 2015-09-11         1
9 2015-10-09         1

Last 10 release dates:
          date  is_wasde
133 2026-03-10         1
134 2026-04-09         1
135 2026-05-12         1
136 2026-06-11         1
137 2026-07-10         1
138 2026-08-12         1
139 2026-09-11         1
140 2026-10-09         1
141 2026-11-10         1
142 2026-12-10         1


## STEP 4.3 — Load Daily Curve Tables

Load the curve tables we built in Step 3.


In [3]:
# Load daily curve tables
corn = pd.read_csv(
    f"{BASE_PATH}/processed/curve_tables/corn_curve.csv",
    parse_dates=["date"]
)
soy = pd.read_csv(
    f"{BASE_PATH}/processed/curve_tables/soy_curve.csv",
    parse_dates=["date"]
)
wheat = pd.read_csv(
    f"{BASE_PATH}/processed/curve_tables/wheat_curve.csv",
    parse_dates=["date"]
)

print("Daily curve tables loaded:")
print(f"Corn:  {corn.shape}")
print(f"Soy:   {soy.shape}")
print(f"Wheat: {wheat.shape}")
print(f"\nDate ranges:")
print(f"Corn:  {corn['date'].min()} to {corn['date'].max()}")
print(f"Soy:   {soy['date'].min()} to {soy['date'].max()}")
print(f"Wheat: {wheat['date'].min()} to {wheat['date'].max()}")


Daily curve tables loaded:
Corn:  (1259, 17)
Soy:   (1260, 17)
Wheat: (1259, 17)

Date ranges:
Corn:  2020-12-09 00:00:00 to 2025-12-09 00:00:00
Soy:   2020-12-09 00:00:00 to 2025-12-10 00:00:00
Wheat: 2020-12-09 00:00:00 to 2025-12-09 00:00:00


## STEP 4.4 — CORN: Load and Align US Corn Ending Stocks

Start with one fundamental series to establish the pattern.


In [4]:
# Load US Corn Ending Stocks
corn_stocks = pd.read_csv(
    f"{BASE_PATH}/raw/fundamentals/wasde/USCORNENDINGSTOCKS(Sheet1).csv"
)

print("=== CORN STOCKS - INSPECTION ===")
print(corn_stocks.head(10))
print("\n")
print(corn_stocks.info())
print("\n")
print(f"Date range: {corn_stocks['Date'].min()} to {corn_stocks['Date'].max()}")


=== CORN STOCKS - INSPECTION ===
         Date  Last Price
0  12/31/2025        2029
1  11/30/2025        2154
2   9/30/2025        2110
3   8/31/2025        2117
4   7/31/2025        1660
5   6/30/2025        1750
6   5/31/2025        1800
7   4/30/2025        1465
8   3/31/2025        1540
9   2/28/2025        1540


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 178 entries, 0 to 177
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   Date        178 non-null    object
 1   Last Price  178 non-null    int64 
dtypes: int64(1), object(1)
memory usage: 2.9+ KB
None


Date range: 1/31/2011 to 9/30/2025


In [5]:
# Standardize corn stocks dataframe
corn_stocks = corn_stocks.rename(columns={
    "Date": "release_date",
    "Last Price": "ending_stocks"
})

# Convert date to datetime
corn_stocks["release_date"] = pd.to_datetime(corn_stocks["release_date"])

# Sort by release date
corn_stocks = corn_stocks.sort_values("release_date").reset_index(drop=True)

# Set release_date as index for merging
corn_stocks_indexed = corn_stocks.set_index("release_date")

print("Corn stocks standardized:")
print(corn_stocks_indexed.head(10))
print(f"\nTotal observations: {len(corn_stocks_indexed)}")


Corn stocks standardized:
              ending_stocks
release_date               
2010-12-31              832
2011-01-31              745
2011-02-28              675
2011-03-31              675
2011-04-30              675
2011-05-31              900
2011-06-30              695
2011-07-31              870
2011-08-31              714
2011-09-30              672

Total observations: 178


### Align Corn Stocks to Daily Data

Key idea: WASDE values are known only on release dates, then forward-filled until next release.


In [6]:
# Merge corn stocks with daily data
# Use merge_asof to find the most recent release date <= each trading date
corn_daily = corn.copy()

# Merge using merge_asof (backward search - finds most recent release <= trading date)
corn_daily = pd.merge_asof(
    corn_daily.sort_values("date"),
    corn_stocks_indexed.sort_index(),
    left_on="date",
    right_index=True,
    direction="backward"
)

# Forward fill to carry values forward until next release
corn_daily["ending_stocks"] = corn_daily["ending_stocks"].ffill()

print("Corn stocks aligned to daily data:")
print(corn_daily[["date", "c1_bid", "ending_stocks"]].head(20))
print("\n...")
print(corn_daily[["date", "c1_bid", "ending_stocks"]].tail(20))


Corn stocks aligned to daily data:
         date  c1_bid  ending_stocks
0  2020-12-09  420.00           1702
1  2020-12-10  419.00           1702
2  2020-12-11  424.50           1702
3  2020-12-14     NaN           1702
4  2020-12-15  424.75           1702
5  2020-12-16  426.25           1702
6  2020-12-17  432.00           1702
7  2020-12-18  436.75           1702
8  2020-12-21  440.00           1702
9  2020-12-22  443.00           1702
10 2020-12-23  447.00           1702
11 2020-12-24  450.50           1702
12 2020-12-28  456.50           1702
13 2020-12-29  466.00           1702
14 2020-12-30  474.00           1702
15 2020-12-31  484.00           1702
16 2021-01-04  484.50           1702
17 2021-01-05  492.50           1702
18 2021-01-06  495.25           1702
19 2021-01-07  494.00           1702

...
           date  c1_bid  ending_stocks
1239 2025-11-11  431.50           2110
1240 2025-11-12  435.00           2110
1241 2025-11-13  442.00           2110
1242 2025-11-14  430.00    

## STEP 4.5 — Create "Surprise" Features

Markets react to changes vs expectations, not absolute levels.


In [7]:
# Create surprise: change in ending stocks
# ↑ ending stocks → bearish (more supply)
# ↓ ending stocks → bullish (less supply)
corn_daily["stocks_change"] = corn_daily["ending_stocks"].diff()
corn_daily["stocks_surprise"] = corn_daily["ending_stocks"] - corn_daily["ending_stocks"].shift(1)

print("Corn stocks surprise computed:")
print(corn_daily[["date", "ending_stocks", "stocks_change", "stocks_surprise"]].head(20))
print("\nSurprise statistics:")
print(corn_daily["stocks_surprise"].describe())


Corn stocks surprise computed:
         date  ending_stocks  stocks_change  stocks_surprise
0  2020-12-09           1702            NaN              NaN
1  2020-12-10           1702            0.0              0.0
2  2020-12-11           1702            0.0              0.0
3  2020-12-14           1702            0.0              0.0
4  2020-12-15           1702            0.0              0.0
5  2020-12-16           1702            0.0              0.0
6  2020-12-17           1702            0.0              0.0
7  2020-12-18           1702            0.0              0.0
8  2020-12-21           1702            0.0              0.0
9  2020-12-22           1702            0.0              0.0
10 2020-12-23           1702            0.0              0.0
11 2020-12-24           1702            0.0              0.0
12 2020-12-28           1702            0.0              0.0
13 2020-12-29           1702            0.0              0.0
14 2020-12-30           1702            0.0           

## STEP 4.6 — Standardize Surprises (Z-Scores)

Raw numbers are not comparable across time. Standardize using rolling standard deviation.


In [8]:
# Standardize surprises using rolling z-score
# Use 24-month rolling window (approximately 2 years of releases)
rolling_std = corn_daily["stocks_surprise"].rolling(window=24, min_periods=1).std()
corn_daily["stocks_surprise_z"] = corn_daily["stocks_surprise"] / (rolling_std + 1e-9)

print("Corn stocks surprise standardized:")
print(corn_daily[["date", "ending_stocks", "stocks_surprise", "stocks_surprise_z"]].head(20))
print("\nStandardized surprise statistics:")
print(corn_daily["stocks_surprise_z"].describe())
print(f"\nPositive z (bearish): {(corn_daily['stocks_surprise_z'] > 0).sum()}")
print(f"Negative z (bullish): {(corn_daily['stocks_surprise_z'] < 0).sum()}")


Corn stocks surprise standardized:
         date  ending_stocks  stocks_surprise  stocks_surprise_z
0  2020-12-09           1702              NaN                NaN
1  2020-12-10           1702              0.0                NaN
2  2020-12-11           1702              0.0                0.0
3  2020-12-14           1702              0.0                0.0
4  2020-12-15           1702              0.0                0.0
5  2020-12-16           1702              0.0                0.0
6  2020-12-17           1702              0.0                0.0
7  2020-12-18           1702              0.0                0.0
8  2020-12-21           1702              0.0                0.0
9  2020-12-22           1702              0.0                0.0
10 2020-12-23           1702              0.0                0.0
11 2020-12-24           1702              0.0                0.0
12 2020-12-28           1702              0.0                0.0
13 2020-12-29           1702              0.0          

## STEP 4.7 — Add More Corn Fundamentals

Add US Corn Supply/Use Yield and Residual.


In [9]:
# Load US Corn Supply/Use Yield
corn_yield = pd.read_csv(
    f"{BASE_PATH}/raw/fundamentals/wasde/USCORNSUPPLYUSEYIELD(Sheet1).csv"
)
corn_yield = corn_yield.rename(columns={"Date": "release_date", "Last Price": "yield"})
corn_yield["release_date"] = pd.to_datetime(corn_yield["release_date"])
corn_yield = corn_yield.sort_values("release_date").set_index("release_date")

# Merge with daily data
corn_daily = pd.merge_asof(
    corn_daily.sort_values("date"),
    corn_yield.sort_index(),
    left_on="date",
    right_index=True,
    direction="backward"
)
corn_daily["yield"] = corn_daily["yield"].ffill()

# Create yield surprise
corn_daily["yield_change"] = corn_daily["yield"].diff()
corn_daily["yield_surprise"] = corn_daily["yield"] - corn_daily["yield"].shift(1)
yield_rolling_std = corn_daily["yield_surprise"].rolling(window=24, min_periods=1).std()
corn_daily["yield_surprise_z"] = corn_daily["yield_surprise"] / (yield_rolling_std + 1e-9)

print("Corn yield added:")
print(corn_daily[["date", "yield", "yield_surprise", "yield_surprise_z"]].head(10))


Corn yield added:
        date  yield  yield_surprise  yield_surprise_z
0 2020-12-09  175.8             NaN               NaN
1 2020-12-10  175.8             0.0               NaN
2 2020-12-11  175.8             0.0               0.0
3 2020-12-14  175.8             0.0               0.0
4 2020-12-15  175.8             0.0               0.0
5 2020-12-16  175.8             0.0               0.0
6 2020-12-17  175.8             0.0               0.0
7 2020-12-18  175.8             0.0               0.0
8 2020-12-21  175.8             0.0               0.0
9 2020-12-22  175.8             0.0               0.0


In [10]:
# Load US Corn Residual
corn_residual = pd.read_csv(
    f"{BASE_PATH}/raw/fundamentals/wasde/USCORNRESIDUAL(Sheet1).csv"
)
corn_residual = corn_residual.rename(columns={"Date": "release_date", "Last Price": "residual"})
corn_residual["release_date"] = pd.to_datetime(corn_residual["release_date"])
corn_residual = corn_residual.sort_values("release_date").set_index("release_date")

# Merge with daily data
corn_daily = pd.merge_asof(
    corn_daily.sort_values("date"),
    corn_residual.sort_index(),
    left_on="date",
    right_index=True,
    direction="backward"
)
corn_daily["residual"] = corn_daily["residual"].ffill()

# Create residual surprise
corn_daily["residual_change"] = corn_daily["residual"].diff()
corn_daily["residual_surprise"] = corn_daily["residual"] - corn_daily["residual"].shift(1)
residual_rolling_std = corn_daily["residual_surprise"].rolling(window=24, min_periods=1).std()
corn_daily["residual_surprise_z"] = corn_daily["residual_surprise"] / (residual_rolling_std + 1e-9)

print("Corn residual added:")
print(corn_daily[["date", "residual", "residual_surprise", "residual_surprise_z"]].head(10))


Corn residual added:
        date  residual  residual_surprise  residual_surprise_z
0 2020-12-09      5700                NaN                  NaN
1 2020-12-10      5700                0.0                  NaN
2 2020-12-11      5700                0.0                  0.0
3 2020-12-14      5700                0.0                  0.0
4 2020-12-15      5700                0.0                  0.0
5 2020-12-16      5700                0.0                  0.0
6 2020-12-17      5700                0.0                  0.0
7 2020-12-18      5700                0.0                  0.0
8 2020-12-21      5700                0.0                  0.0
9 2020-12-22      5700                0.0                  0.0


## SOYBEAN: Add Fundamentals

Add soybean ending stocks, exports, and yield.


In [11]:
# Load Soybean Ending Stocks
soy_stocks = pd.read_csv(
    f"{BASE_PATH}/raw/fundamentals/wasde/SOYBEANENDINGSTOCK(Sheet1).csv"
)
soy_stocks = soy_stocks.rename(columns={"Date": "release_date", "Last Price": "ending_stocks"})
soy_stocks["release_date"] = pd.to_datetime(soy_stocks["release_date"])
soy_stocks = soy_stocks.sort_values("release_date").set_index("release_date")

# Merge with daily data
soy_daily = pd.merge_asof(
    soy.sort_values("date"),
    soy_stocks.sort_index(),
    left_on="date",
    right_index=True,
    direction="backward"
)
soy_daily["ending_stocks"] = soy_daily["ending_stocks"].ffill()

# Create surprise
soy_daily["stocks_change"] = soy_daily["ending_stocks"].diff()
soy_daily["stocks_surprise"] = soy_daily["ending_stocks"] - soy_daily["ending_stocks"].shift(1)
stocks_rolling_std = soy_daily["stocks_surprise"].rolling(window=24, min_periods=1).std()
soy_daily["stocks_surprise_z"] = soy_daily["stocks_surprise"] / (stocks_rolling_std + 1e-9)

print("Soybean ending stocks added:")
print(soy_daily[["date", "s1_bid", "ending_stocks", "stocks_surprise_z"]].head(10))


Soybean ending stocks added:
        date   s1_bid  ending_stocks  stocks_surprise_z
0 2020-12-09  1158.25            190                NaN
1 2020-12-10  1154.00            190                NaN
2 2020-12-11  1156.00            190                0.0
3 2020-12-14  1168.50            190                0.0
4 2020-12-15  1182.50            190                0.0
5 2020-12-16  1181.75            190                0.0
6 2020-12-17  1204.50            190                0.0
7 2020-12-18  1217.00            190                0.0
8 2020-12-21  1241.50            190                0.0
9 2020-12-22  1247.75            190                0.0


In [12]:
# Load Soybean Exports
soy_exports = pd.read_csv(
    f"{BASE_PATH}/raw/fundamentals/wasde/SOYBEANSEXPORT(Sheet1).csv"
)
soy_exports = soy_exports.rename(columns={"Date": "release_date", "Last Price": "exports"})
soy_exports["release_date"] = pd.to_datetime(soy_exports["release_date"])
soy_exports = soy_exports.sort_values("release_date").set_index("release_date")

# Merge with daily data
soy_daily = pd.merge_asof(
    soy_daily.sort_values("date"),
    soy_exports.sort_index(),
    left_on="date",
    right_index=True,
    direction="backward"
)
soy_daily["exports"] = soy_daily["exports"].ffill()

# Create surprise
soy_daily["exports_change"] = soy_daily["exports"].diff()
soy_daily["exports_surprise"] = soy_daily["exports"] - soy_daily["exports"].shift(1)
exports_rolling_std = soy_daily["exports_surprise"].rolling(window=24, min_periods=1).std()
soy_daily["exports_surprise_z"] = soy_daily["exports_surprise"] / (exports_rolling_std + 1e-9)

print("Soybean exports added:")
print(soy_daily[["date", "exports", "exports_surprise_z"]].head(10))


Soybean exports added:
        date  exports  exports_surprise_z
0 2020-12-09     2200                 NaN
1 2020-12-10     2200                 NaN
2 2020-12-11     2200                 0.0
3 2020-12-14     2200                 0.0
4 2020-12-15     2200                 0.0
5 2020-12-16     2200                 0.0
6 2020-12-17     2200                 0.0
7 2020-12-18     2200                 0.0
8 2020-12-21     2200                 0.0
9 2020-12-22     2200                 0.0


In [13]:
# Load Soybean Yield
soy_yield = pd.read_csv(
    f"{BASE_PATH}/raw/fundamentals/wasde/SOYBEANYIELD(Sheet1).csv"
)
soy_yield = soy_yield.rename(columns={"Date": "release_date", "Last Price": "yield"})
soy_yield["release_date"] = pd.to_datetime(soy_yield["release_date"])
soy_yield = soy_yield.sort_values("release_date").set_index("release_date")

# Merge with daily data
soy_daily = pd.merge_asof(
    soy_daily.sort_values("date"),
    soy_yield.sort_index(),
    left_on="date",
    right_index=True,
    direction="backward"
)
soy_daily["yield"] = soy_daily["yield"].ffill()

# Create surprise
soy_daily["yield_change"] = soy_daily["yield"].diff()
soy_daily["yield_surprise"] = soy_daily["yield"] - soy_daily["yield"].shift(1)
yield_rolling_std = soy_daily["yield_surprise"].rolling(window=24, min_periods=1).std()
soy_daily["yield_surprise_z"] = soy_daily["yield_surprise"] / (yield_rolling_std + 1e-9)

print("Soybean yield added:")
print(soy_daily[["date", "yield", "yield_surprise_z"]].head(10))


Soybean yield added:
        date  yield  yield_surprise_z
0 2020-12-09   50.7               NaN
1 2020-12-10   50.7               NaN
2 2020-12-11   50.7               0.0
3 2020-12-14   50.7               0.0
4 2020-12-15   50.7               0.0
5 2020-12-16   50.7               0.0
6 2020-12-17   50.7               0.0
7 2020-12-18   50.7               0.0
8 2020-12-21   50.7               0.0
9 2020-12-22   50.7               0.0


## WHEAT: Add Fundamentals

Add wheat production, total ending stocks, and world ending stocks.


In [14]:
# Load Wheat Production
wheat_prod = pd.read_csv(
    f"{BASE_PATH}/raw/fundamentals/wasde/WHEATPRODUCTION(Sheet1).csv"
)
wheat_prod = wheat_prod.rename(columns={"Date": "release_date", "Last Price": "production"})
wheat_prod["release_date"] = pd.to_datetime(wheat_prod["release_date"])
wheat_prod = wheat_prod.sort_values("release_date").set_index("release_date")

# Merge with daily data
wheat_daily = pd.merge_asof(
    wheat.sort_values("date"),
    wheat_prod.sort_index(),
    left_on="date",
    right_index=True,
    direction="backward"
)
wheat_daily["production"] = wheat_daily["production"].ffill()

# Create surprise
wheat_daily["production_change"] = wheat_daily["production"].diff()
wheat_daily["production_surprise"] = wheat_daily["production"] - wheat_daily["production"].shift(1)
prod_rolling_std = wheat_daily["production_surprise"].rolling(window=24, min_periods=1).std()
wheat_daily["production_surprise_z"] = wheat_daily["production_surprise"] / (prod_rolling_std + 1e-9)

print("Wheat production added:")
print(wheat_daily[["date", "w1_bid", "production", "production_surprise_z"]].head(10))


Wheat production added:
        date  w1_bid  production  production_surprise_z
0 2020-12-09  575.00      1826.0                    NaN
1 2020-12-10  553.00      1826.0                    NaN
2 2020-12-11  553.00      1826.0                    0.0
3 2020-12-14     NaN      1826.0                    0.0
4 2020-12-15  601.00      1826.0                    0.0
5 2020-12-16  598.00      1826.0                    0.0
6 2020-12-17  609.50      1826.0                    0.0
7 2020-12-18  607.50      1826.0                    0.0
8 2020-12-21  611.25      1826.0                    0.0
9 2020-12-22  615.75      1826.0                    0.0


In [15]:
# Load Total Wheat Ending Stocks
wheat_stocks_total = pd.read_csv(
    f"{BASE_PATH}/raw/fundamentals/wasde/TOTALWHEATENDINGSTOCKS(Sheet1).csv"
)
wheat_stocks_total = wheat_stocks_total.rename(columns={"Date": "release_date", "Last Price": "ending_stocks_total"})
wheat_stocks_total["release_date"] = pd.to_datetime(wheat_stocks_total["release_date"])
wheat_stocks_total = wheat_stocks_total.sort_values("release_date").set_index("release_date")

# Merge with daily data
wheat_daily = pd.merge_asof(
    wheat_daily.sort_values("date"),
    wheat_stocks_total.sort_index(),
    left_on="date",
    right_index=True,
    direction="backward"
)
wheat_daily["ending_stocks_total"] = wheat_daily["ending_stocks_total"].ffill()

# Create surprise
wheat_daily["stocks_total_change"] = wheat_daily["ending_stocks_total"].diff()
wheat_daily["stocks_total_surprise"] = wheat_daily["ending_stocks_total"] - wheat_daily["ending_stocks_total"].shift(1)
stocks_total_rolling_std = wheat_daily["stocks_total_surprise"].rolling(window=24, min_periods=1).std()
wheat_daily["stocks_total_surprise_z"] = wheat_daily["stocks_total_surprise"] / (stocks_total_rolling_std + 1e-9)

print("Wheat total ending stocks added:")
print(wheat_daily[["date", "ending_stocks_total", "stocks_total_surprise_z"]].head(10))


Wheat total ending stocks added:
        date  ending_stocks_total  stocks_total_surprise_z
0 2020-12-09                  877                      NaN
1 2020-12-10                  877                      NaN
2 2020-12-11                  877                      0.0
3 2020-12-14                  877                      0.0
4 2020-12-15                  877                      0.0
5 2020-12-16                  877                      0.0
6 2020-12-17                  877                      0.0
7 2020-12-18                  877                      0.0
8 2020-12-21                  877                      0.0
9 2020-12-22                  877                      0.0


In [16]:
# Load World Wheat Ending Stocks
wheat_stocks_world = pd.read_csv(
    f"{BASE_PATH}/raw/fundamentals/wasde/WORLDWHEATENGINSTOCKS(Sheet1).csv"
)
wheat_stocks_world = wheat_stocks_world.rename(columns={"Date": "release_date", "Last Price": "ending_stocks_world"})
wheat_stocks_world["release_date"] = pd.to_datetime(wheat_stocks_world["release_date"])
wheat_stocks_world = wheat_stocks_world.sort_values("release_date").set_index("release_date")

# Merge with daily data
wheat_daily = pd.merge_asof(
    wheat_daily.sort_values("date"),
    wheat_stocks_world.sort_index(),
    left_on="date",
    right_index=True,
    direction="backward"
)
wheat_daily["ending_stocks_world"] = wheat_daily["ending_stocks_world"].ffill()

# Create surprise
wheat_daily["stocks_world_change"] = wheat_daily["ending_stocks_world"].diff()
wheat_daily["stocks_world_surprise"] = wheat_daily["ending_stocks_world"] - wheat_daily["ending_stocks_world"].shift(1)
stocks_world_rolling_std = wheat_daily["stocks_world_surprise"].rolling(window=24, min_periods=1).std()
wheat_daily["stocks_world_surprise_z"] = wheat_daily["stocks_world_surprise"] / (stocks_world_rolling_std + 1e-9)

print("Wheat world ending stocks added:")
print(wheat_daily[["date", "ending_stocks_world", "stocks_world_surprise_z"]].head(10))


Wheat world ending stocks added:
        date  ending_stocks_world  stocks_world_surprise_z
0 2020-12-09               320.45                      NaN
1 2020-12-10               320.45                      NaN
2 2020-12-11               320.45                      0.0
3 2020-12-14               320.45                      0.0
4 2020-12-15               320.45                      0.0
5 2020-12-16               320.45                      0.0
6 2020-12-17               320.45                      0.0
7 2020-12-18               320.45                      0.0
8 2020-12-21               320.45                      0.0
9 2020-12-22               320.45                      0.0


## STEP 4.8 — Sanity Checks

Verify that surprises align with WASDE release dates and show expected patterns.


In [17]:
# CORN: Sanity checks
print("=== CORN WASDE ALIGNMENT ===")
print(f"\nShape: {corn_daily.shape}")
print(f"\nColumns with 'surprise': {[col for col in corn_daily.columns if 'surprise' in col]}")
print(f"\nDate range: {corn_daily['date'].min()} to {corn_daily['date'].max()}")
print("\nSample of corn data with surprises:")
print(corn_daily[["date", "c1_bid", "ending_stocks", "stocks_surprise_z", "yield_surprise_z"]].tail(20))


=== CORN WASDE ALIGNMENT ===

Shape: (1259, 29)

Columns with 'surprise': ['stocks_surprise', 'stocks_surprise_z', 'yield_surprise', 'yield_surprise_z', 'residual_surprise', 'residual_surprise_z']

Date range: 2020-12-09 00:00:00 to 2025-12-09 00:00:00

Sample of corn data with surprises:
           date  c1_bid  ending_stocks  stocks_surprise_z  yield_surprise_z
1239 2025-11-11  431.50           2110           0.000000          0.000000
1240 2025-11-12  435.00           2110           0.000000          0.000000
1241 2025-11-13  442.00           2110           0.000000          0.000000
1242 2025-11-14  430.00           2110           0.000000          0.000000
1243 2025-11-17  434.75           2110           0.000000          0.000000
1244 2025-11-18  435.50           2110           0.000000          0.000000
1245 2025-11-19  429.75           2110           0.000000          0.000000
1246 2025-11-20  426.25           2110           0.000000          0.000000
1247 2025-11-21  425.75   

In [18]:
# SOYBEAN: Sanity checks
print("=== SOYBEAN WASDE ALIGNMENT ===")
print(f"\nShape: {soy_daily.shape}")
print(f"\nColumns with 'surprise': {[col for col in soy_daily.columns if 'surprise' in col]}")
print(f"\nDate range: {soy_daily['date'].min()} to {soy_daily['date'].max()}")
print("\nSample of soy data with surprises:")
print(soy_daily[["date", "s1_bid", "ending_stocks", "stocks_surprise_z", "exports_surprise_z", "yield_surprise_z"]].tail(20))


=== SOYBEAN WASDE ALIGNMENT ===

Shape: (1260, 29)

Columns with 'surprise': ['stocks_surprise', 'stocks_surprise_z', 'exports_surprise', 'exports_surprise_z', 'yield_surprise', 'yield_surprise_z']

Date range: 2020-12-09 00:00:00 to 2025-12-10 00:00:00

Sample of soy data with surprises:
           date   s1_bid  ending_stocks  stocks_surprise_z  \
1240 2025-11-12  1108.25            300           0.000000   
1241 2025-11-13  1017.50            300           0.000000   
1242 2025-11-14      NaN            300           0.000000   
1243 2025-11-17  1157.00            300           0.000000   
1244 2025-11-18  1150.00            300           0.000000   
1245 2025-11-19  1135.00            300           0.000000   
1246 2025-11-20  1123.00            300           0.000000   
1247 2025-11-21  1125.50            300           0.000000   
1248 2025-11-24  1120.75            300           0.000000   
1249 2025-11-25  1124.00            300           0.000000   
1250 2025-11-26  1131.75    

In [19]:
# WHEAT: Sanity checks
print("=== WHEAT WASDE ALIGNMENT ===")
print(f"\nShape: {wheat_daily.shape}")
print(f"\nColumns with 'surprise': {[col for col in wheat_daily.columns if 'surprise' in col]}")
print(f"\nDate range: {wheat_daily['date'].min()} to {wheat_daily['date'].max()}")
print("\nSample of wheat data with surprises:")
print(wheat_daily[["date", "w1_bid", "production", "production_surprise_z", "stocks_total_surprise_z"]].tail(20))


=== WHEAT WASDE ALIGNMENT ===

Shape: (1259, 29)

Columns with 'surprise': ['production_surprise', 'production_surprise_z', 'stocks_total_surprise', 'stocks_total_surprise_z', 'stocks_world_surprise', 'stocks_world_surprise_z']

Date range: 2020-12-09 00:00:00 to 2025-12-09 00:00:00

Sample of wheat data with surprises:
           date  w1_bid  production  production_surprise_z  \
1239 2025-11-11  536.00      1927.0               0.000000   
1240 2025-11-12  536.50      1927.0               0.000000   
1241 2025-11-13  535.25      1927.0               0.000000   
1242 2025-11-14  526.50      1927.0               0.000000   
1243 2025-11-17  544.75      1927.0               0.000000   
1244 2025-11-18  546.50      1927.0               0.000000   
1245 2025-11-19  537.00      1927.0               0.000000   
1246 2025-11-20  527.75      1927.0               0.000000   
1247 2025-11-21  529.50      1927.0               0.000000   
1248 2025-11-24  522.50      1927.0               0.000000

In [20]:
# Save all WASDE-aligned tables
corn_daily.to_csv(
    f"{BASE_PATH}/processed/wasde_aligned/corn_wasde.csv",
    index=False
)
soy_daily.to_csv(
    f"{BASE_PATH}/processed/wasde_aligned/soy_wasde.csv",
    index=False
)
wheat_daily.to_csv(
    f"{BASE_PATH}/processed/wasde_aligned/wheat_wasde.csv",
    index=False
)

print("✅ All WASDE-aligned tables saved:")
print(f"  - {BASE_PATH}/processed/wasde_aligned/corn_wasde.csv")
print(f"  - {BASE_PATH}/processed/wasde_aligned/soy_wasde.csv")
print(f"  - {BASE_PATH}/processed/wasde_aligned/wheat_wasde.csv")
print("\nFinal shapes:")
print(f"  Corn:  {corn_daily.shape}")
print(f"  Soy:   {soy_daily.shape}")
print(f"  Wheat: {wheat_daily.shape}")
print("\nSummary of features added:")
print("  CORN:")
print("    - ending_stocks, stocks_surprise, stocks_surprise_z")
print("    - yield, yield_surprise, yield_surprise_z")
print("    - residual, residual_surprise, residual_surprise_z")
print("  SOYBEAN:")
print("    - ending_stocks, stocks_surprise, stocks_surprise_z")
print("    - exports, exports_surprise, exports_surprise_z")
print("    - yield, yield_surprise, yield_surprise_z")
print("  WHEAT:")
print("    - production, production_surprise, production_surprise_z")
print("    - ending_stocks_total, stocks_total_surprise, stocks_total_surprise_z")
print("    - ending_stocks_world, stocks_world_surprise, stocks_world_surprise_z")


✅ All WASDE-aligned tables saved:
  - /Users/aryansinha/Desktop/WASDA/processed/wasde_aligned/corn_wasde.csv
  - /Users/aryansinha/Desktop/WASDA/processed/wasde_aligned/soy_wasde.csv
  - /Users/aryansinha/Desktop/WASDA/processed/wasde_aligned/wheat_wasde.csv

Final shapes:
  Corn:  (1259, 29)
  Soy:   (1260, 29)
  Wheat: (1259, 29)

Summary of features added:
  CORN:
    - ending_stocks, stocks_surprise, stocks_surprise_z
    - yield, yield_surprise, yield_surprise_z
    - residual, residual_surprise, residual_surprise_z
  SOYBEAN:
    - ending_stocks, stocks_surprise, stocks_surprise_z
    - exports, exports_surprise, exports_surprise_z
    - yield, yield_surprise, yield_surprise_z
  WHEAT:
    - production, production_surprise, production_surprise_z
    - ending_stocks_total, stocks_total_surprise, stocks_total_surprise_z
    - ending_stocks_world, stocks_world_surprise, stocks_world_surprise_z


## Summary

All three commodities now have:
- ✅ WASDE fundamentals aligned to release dates
- ✅ Surprise features (changes vs previous release)
- ✅ Standardized surprises (z-scores) for comparability

**Key insights:**
- Positive surprise z-scores → bearish (more supply, less demand)
- Negative surprise z-scores → bullish (less supply, more demand)
- Surprises are forward-filled until next WASDE release

**Next steps:**
- These tables are ready for feature engineering and ML
- Can now analyze how market reacts to WASDE surprises
